# Demo 2: the solving agent

Watch the agent solve cryptic clues the way an expert does: it reads the clue, calls
tools (anagram and hidden-word checks, past definitions, abbreviations, indicators),
and hands in a **worksheet**: a full parse saying which words are the definition,
indicator, fodder and link words. Then **code checks that worksheet** and decides
the verdict:

- **confirmed**: definition at one end, wordplay verified mechanically, every word has
  a role, answer is a real word of the right length. Code checked all of it.
- **pencilled**: the answer fits, but part of it can't be proven by code yet (e.g. a
  charade built from synonyms, or a double definition).
- **unsure**: the answer doesn't even fit the enumeration or the crossing letters.

**Setup (once)**, in a terminal at the repo root:

```bash
uv sync --group demo          # notebook kernel + pandas
uv run cryptic-agent ingest   # word list + lexicon (about a minute)
```

and put your Groq key in `.env` as `GROQ_API_KEY=...`. The free tier allows 8,000
tokens a minute; a solve uses roughly 3,000 to 7,000, so the client pauses between
solves when needed (you'll see "waiting" messages).

In [ ]:
import logging
import os
import textwrap
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from cryptic_agent.agent.solver import Solver, SolveResult, Step
from cryptic_agent.agent.tools import Toolbox
from cryptic_agent.extraction.table_parser import parse_clue_table
from cryptic_agent.jsonl import read_jsonl
from cryptic_agent.lexicon.store import Lexicon
from cryptic_agent.llm.client import GroqClient
from cryptic_agent.models import RawPost
from cryptic_agent.tools.dictionary import Dictionary


def find_repo_root(start: Path) -> Path:
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("run this from inside the cryptic-agent repo")


ROOT = find_repo_root(Path.cwd())
load_dotenv(ROOT / ".env")
logging.basicConfig(level=logging.WARNING)
logging.getLogger("cryptic_agent.llm.client").setLevel(logging.INFO)  # show rate-limit waits

# The tools need the word list and lexicon from `cryptic-agent ingest`.
os.environ.setdefault("CRYPTIC_AGENT_DATA_DIR", str(ROOT / "data"))
toolbox = Toolbox(Dictionary.load(), Lexicon())
llm = GroqClient()
print(f"model: {llm.model}; {len(toolbox.tools)} tools: {', '.join(toolbox.tools)}")

## 1. Watch one solve
Every thought (the model's private reasoning, shortened), every tool call and its
result, then the worksheet and the code checks.

In [ ]:
def print_step(step: Step) -> None:
    if step.kind == "thought":
        print("💭", textwrap.shorten(step.text, 400))
    elif step.kind == "tool":
        args = ", ".join(f"{k}={v!r}" for k, v in step.arguments.items())
        print(f"🔧 {step.tool}({args})\n   → {textwrap.shorten(step.result, 250)}")


def report(result: SolveResult) -> None:
    if result.verdict is None or result.worksheet is None:
        print("No valid worksheet:", result.error)
        return
    ws, v = result.worksheet, result.verdict
    print(f"\nANSWER: {v.answer}   [{v.status.upper()}]")
    for d in ws.definitions:
        print(f"  definition: {d.text!r} ({d.position})")
    for w in ws.wordplay:
        indicator = f", indicator {w.indicator!r}" if w.indicator else ""
        print(f"  wordplay:   {w.mechanism} of {w.fodder!r}{indicator} → {w.produces}")
    if ws.link_words:
        print(f"  link words: {ws.link_words}")
    marks = {True: "✓", False: "✗", None: "?"}
    print("checks:")
    for c in v.checks:
        print(f"  {marks[c.passed]} {c.name}: {c.detail}")
    print(f"cost: {result.usage.total_tokens:,} tokens")


def watch(clue: str, enumeration: str, pattern: str | None = None) -> SolveResult:
    print(f"Clue: {clue} ({enumeration})" + (f"   letters: {pattern}" if pattern else ""))
    result = Solver(llm, toolbox, on_step=print_step).solve(clue, enumeration, pattern=pattern)
    report(result)
    return result


watch("Cook using mere sweet desserts", "9");

## 2. Try your own clue
Change the clue and enumeration and run the cell.

In [ ]:
watch("Steal small amount of salt, say", "5");

## 3. With crossing letters
In a real grid, crossing answers give you some letters. The agent is told them, and
the verdict checks the answer against them: a mismatch can never be "confirmed".

In [ ]:
watch("Mountain observed in severe storm", "7", pattern="E?E?E?T");

## 4. A small batch of clues it has never seen
One clue of each kind from recent Quick Cryptic puzzles (2025 onwards, newer than the
lexicon's data), scored against the blogger's answer. Takes a few minutes because of
the free tier's rate limit.

In [ ]:
posts = read_jsonl(ROOT / "data/raw/guardian_quick-cryptic.jsonl", RawPost)
recent = [c for p in posts if p.date.year >= 2025 for c in parse_clue_table(p)]
kinds = ["anagram", "hidden_word", "acrostic", "reversal", "double_definition", "charade"]
batch = [next(c for c in recent if c.type_hints == [k] and len(c.answer) >= 5) for k in kinds]

rows = []
for clue in batch:
    result = Solver(llm, toolbox).solve(clue.clue_text, clue.enumeration)
    rows.append(
        {
            "kind": clue.type_hints[0],
            "clue": f"{clue.clue_text} ({clue.enumeration})",
            "expected": clue.answer,
            "answer": result.answer,
            "right": result.answer == clue.answer,
            "verdict": result.verdict.status if result.verdict else "error",
            "tokens": result.usage.total_tokens,
        }
    )

pd.set_option("display.max_colwidth", 60)
scores = pd.DataFrame(rows)
display(scores)
print(f"right: {scores.right.sum()}/{len(scores)}")
print(
    "confirmed but wrong (the number that must stay 0):",
    int(((scores.verdict == "confirmed") & ~scores.right).sum()),
)
waited = llm.totals.waited_seconds
print(f"total: {llm.totals.requests} requests, waited {waited:.0f}s for rate limits")